In [1]:
# from sklearn.model_selection import train_test_split


# X_train , X_test , y_train , y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [2]:
import numpy as np 
import pandas as pd
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split , cross_val_score ,KFold ,StratifiedKFold ,cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

In [3]:
iris = load_iris()
X = iris.data
y = iris.target

print(X.shape)
print(y.shape)

(150, 4)
(150,)


In [4]:
model = DecisionTreeClassifier(max_depth=4,random_state=42)

score = cross_val_score(model , X,y, cv=5 , scoring="accuracy")

print("score Accuracy = ",score)
print("Average accuracy = ",score.mean())
print("std deviation :=  ",score.std())

score Accuracy =  [0.96666667 0.96666667 0.9        0.93333333 1.        ]
Average accuracy =  0.9533333333333334
std deviation :=   0.03399346342395189


In [5]:
# KFold - Manual Loop
kf = KFold(n_splits=5,shuffle=True,random_state=42)

accuracies = []

for fold,(train_idx,test_idx) in enumerate(kf.split(X),1):
    X_train , X_test = X[train_idx],X[test_idx]
    y_train , y_test = y[train_idx],y[test_idx]

    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test,y_pred)
    accuracies.append(acc)
    print(f'Fold {fold} accuracy = {acc:.4f}')

Fold 1 accuracy = 1.0000
Fold 2 accuracy = 0.9667
Fold 3 accuracy = 0.9333
Fold 4 accuracy = 0.9333
Fold 5 accuracy = 0.9333


In [6]:
print(f"average = {np.mean(accuracies)}")

average = 0.9533333333333335


In [7]:
print(f"std = {np.std(accuracies)}")

std = 0.02666666666666666


In [11]:
# classification -> StratifiedKFold Class balance 3 
skf = StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
scores = cross_val_score(model,X,y,cv=skf,scoring = 'accuracy')
print(f"Stratified 5 fold CV {score}")
print(f"mean = {scores.mean()}")

Stratified 5 fold CV [0.96666667 0.96666667 0.9        0.93333333 1.        ]
mean = 0.9533333333333335


In [14]:
# cross validate -> Multiple Metrics + Train/Score time

scoring = ['accuracy','precision_macro','recall_macro','f1_macro']
cv_result = cross_validate(
    model,
    X,y,
    cv=5,
    scoring=scoring,
    return_train_score = True,
    n_jobs = 1 # faster multi-core
)

In [17]:
print(f"keys in result : {cv_result.keys()}")
print(f"Test accuracy (mean) {cv_result['test_accuracy'].mean()}")
print(f"Train accuracy (mean) {cv_result['train_accuracy'].mean()}")


keys in result : dict_keys(['fit_time', 'score_time', 'test_accuracy', 'train_accuracy', 'test_precision_macro', 'train_precision_macro', 'test_recall_macro', 'train_recall_macro', 'test_f1_macro', 'train_f1_macro'])
Test accuracy (mean) 0.9533333333333334
Train accuracy (mean) 0.9883333333333333


In [18]:
print("Overfitting GAP - ",cv_result['train_accuracy'].mean() - cv_result['test_accuracy'].mean())

Overfitting GAP -  0.03499999999999992


In [24]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'max_depth':[2,3,4,5,6,7,None],
    'min_samples_split':[2,5,10]
}

grid = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    scoring ='accuracy',
    cv=5,
    n_jobs=-1
)
grid.fit(X,y)

GridSearchCV(cv=5, estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [2, 3, 4, 5, 6, 7, None],
                         'min_samples_split': [2, 5, 10]},
             scoring='accuracy')

In [25]:
print('Best Params : - ',grid.best_params_)
print('Best CV Score : - ',grid.best_score_)

Best Params : -  {'max_depth': 3, 'min_samples_split': 2}
Best CV Score : -  0.9733333333333334
